In [ ]:
# PmodI2S2HdmiGuiOneCell.ipynb
# One-cell Notebook that runs the rotary-encoder HDMI GUI while the
# Pmod I2S2 master is held in mode 2 (ADC -> AudioLab DSP -> DAC, D49).
#
# WHAT THIS CELL DOES:
#   - Spawns `scripts/run_encoder_hdmi_gui.py --live-apply --skip-rat
#     --pmod-mode dsp` as a sudo subprocess. The runner loads
#     AudioLabOverlay, starts the integrated HDMI back end, builds the
#     compact-v2 GUI, polls the encoder IP, and writes the Pmod I2S2
#     MODE register to 2 (`dsp`) at startup / 3 (`mute`) at shutdown.
#   - Streams the runner's stdout to an `ipywidgets.Output` log so the
#     Notebook stays interactive (the runner loop runs in the
#     subprocess; this Notebook cell returns immediately).
#   - Exposes Start / Stop / Panic-Mute / Set DSP / Refresh status /
#     Show command buttons. Panic-Mute and Stop both SIGTERM the
#     subprocess (the runner mutes the Pmod DAC during shutdown).
#     Set DSP and Refresh shell out to `scripts/pmod_i2s2_mode.py`
#     which uses `pynq.Overlay(... download=False)` -- no codec
#     reconfig, no bit re-download.
#
# HARDWARE WIRING (do this BEFORE you press Start, music level LOW):
#   - Pmod I2S2 plugged into PMOD JB (PMOD JB is Pmod-I2S2-only, D48).
#   - DISCONNECT the on-module Line Out <-> Line In 3.5 mm jumper.
#   - External source (audio IF OUT / guitar pedal output / phone
#     headphone OUT) -> Pmod I2S2 Line In, source level at MINIMUM.
#   - Pmod I2S2 Line Out -> audio IF IN / powered speakers / headphone
#     amp; do NOT plug Line Out back into Line In.
#
# ROTARY ENCODER UX (live HDMI GUI, see ENCODER_GUI_CONTROL_SPEC.md):
#   - Encoder 0 rotate            -> select effect
#   - Encoder 0 short press       -> toggle current effect ON / OFF
#   - Encoder 1 rotate            -> select focused knob
#   - Encoder 1 hold + rotate     -> cycle model (OD / DIST / AMP / CAB)
#   - Encoder 2 rotate            -> change focused knob value (live apply)
#
# RUNTIME NOTES:
#   - This Notebook does NOT load AudioLabOverlay itself, so the
#     rgb2dvi PLL at 40 MHz (D25) is loaded by the runner only once.
#   - Panic-Mute / Stop SIGTERM the runner; on exit the runner writes
#     MODE=3 (mute) and stops the HDMI back end, so the LCD goes
#     blank-ish but the FPGA stays programmed.
#   - Re-execute this cell (or click Start) to spawn a new runner;
#     the smart-attach pattern in the runner means a second startup
#     in the same boot is safe.
#   - Running two instances of the runner at once will fight over
#     the encoder IP / HDMI VDMA. The Notebook always Stops the old
#     runner before starting a new one.

import os
import shlex
import signal
import subprocess
import sys
import threading
import time

import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
RUNNER_PATH = os.path.join(PROJECT_ROOT, "scripts",
                           "run_encoder_hdmi_gui.py")
MODE_HELPER_PATH = os.path.join(PROJECT_ROOT, "scripts",
                                "pmod_i2s2_mode.py")

RUNNER_CMD = [
    "sudo", "env", "PYTHONPATH=%s" % PROJECT_ROOT,
    "python3", "-u", RUNNER_PATH,
    "--live-apply", "--skip-rat", "--pmod-mode", "dsp",
]

MODE_HELPER_PREFIX = [
    "sudo", "env", "PYTHONPATH=%s" % PROJECT_ROOT,
    "python3", "-u", MODE_HELPER_PATH,
]

MAX_LOG_LINES = 400

_proc_lock = threading.Lock()
_proc = {"handle": None, "reader": None}
_log_buf = []

log_out = widgets.Output(layout=widgets.Layout(
    border="1px solid #888", padding="6px",
    height="260px", overflow="auto"))
status_out = widgets.Output(layout=widgets.Layout(
    border="1px solid #444", padding="6px",
    height="260px", overflow="auto"))


def _log(msg):
    ts = time.strftime("%H:%M:%S")
    line = "[" + ts + "] " + str(msg)
    _log_buf.append(line)
    while len(_log_buf) > MAX_LOG_LINES:
        _log_buf.pop(0)
    with log_out:
        clear_output(wait=True)
        print("\n".join(_log_buf))


def _reader_loop(proc):
    try:
        for raw in iter(proc.stdout.readline, b""):
            try:
                line = raw.decode("utf-8", errors="replace").rstrip()
            except Exception:
                line = repr(raw)
            if line:
                _log("runner: " + line)
    except Exception as exc:
        _log("runner reader error: " + repr(exc))
    finally:
        rc = proc.poll()
        _log("runner exited (rc=%s)" % rc)


def _kill_orphan_runners(label="cleanup"):
    """SIGTERM any `run_encoder_hdmi_gui.py` process this Notebook does not own.

    When the cell is re-executed the module-level `_proc` dict is reset
    to a fresh handle, but any previously-launched runner subprocess is
    still alive at the OS level. Without cleanup, `_start_runner_locked`
    would happily Popen a second runner so two GUIs would fight over
    the encoder IP / HDMI VDMA / Pmod MODE register. This helper pgreps
    for the runner script path, excludes our own kernel pid and (if
    alive) the currently-tracked handle pid, and SIGTERMs the rest.
    Holdouts get a SIGKILL pass. Uses sudo because the runner is
    spawned with `sudo env ...`.
    """
    h = _proc.get("handle")
    handle_pid = h.pid if (h is not None and h.poll() is None) else None
    try:
        out = subprocess.run(
            ["pgrep", "-f", "scripts/run_encoder_hdmi_gui.py"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=5)
    except Exception as exc:
        _log(label + ": pgrep failed: " + repr(exc))
        return 0
    if out.returncode != 0:
        return 0
    pids = []
    for tok in out.stdout.decode("utf-8", errors="replace").split():
        try:
            p = int(tok)
        except ValueError:
            continue
        if p == os.getpid() or p == handle_pid:
            continue
        pids.append(p)
    if not pids:
        return 0
    _log(label + ": SIGTERM orphan runner pid(s) " +
         ",".join(str(p) for p in pids))
    try:
        subprocess.run(["sudo", "kill", "-TERM"] + [str(p) for p in pids],
                       timeout=10)
    except Exception as exc:
        _log(label + ": SIGTERM call failed: " + repr(exc))
    deadline = time.time() + 4.0
    while time.time() < deadline:
        chk = subprocess.run(
            ["pgrep", "-f", "scripts/run_encoder_hdmi_gui.py"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=5)
        if chk.returncode != 0:
            break
        survivors = []
        for tok in chk.stdout.decode("utf-8", errors="replace").split():
            try:
                p = int(tok)
            except ValueError:
                continue
            if p == os.getpid() or p == handle_pid:
                continue
            if p in pids:
                survivors.append(p)
        if not survivors:
            break
        time.sleep(0.5)
    chk = subprocess.run(
        ["pgrep", "-f", "scripts/run_encoder_hdmi_gui.py"],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=5)
    holdouts = []
    if chk.returncode == 0:
        for tok in chk.stdout.decode("utf-8", errors="replace").split():
            try:
                p = int(tok)
            except ValueError:
                continue
            if p == os.getpid() or p == handle_pid:
                continue
            if p in pids:
                holdouts.append(p)
    if holdouts:
        _log(label + ": SIGKILL holdouts " +
             ",".join(str(p) for p in holdouts))
        try:
            subprocess.run(["sudo", "kill", "-KILL"] +
                           [str(p) for p in holdouts], timeout=10)
        except Exception as exc:
            _log(label + ": SIGKILL call failed: " + repr(exc))
    return len(pids)


def _is_running():
    h = _proc["handle"]
    return h is not None and h.poll() is None


def _start_runner_locked():
    if _is_running():
        _log("runner already running (pid=%d); skipping start."
             % _proc["handle"].pid)
        return
    # Reap stale runners from previous cell executions before spawning.
    _kill_orphan_runners(label="pre-start")
    if not os.path.exists(RUNNER_PATH):
        _log("ERROR: runner not found at " + RUNNER_PATH +
             " -- is the project deployed under /home/xilinx?")
        return
    _log("starting runner: " + " ".join(shlex.quote(c) for c in RUNNER_CMD))
    try:
        proc = subprocess.Popen(
            RUNNER_CMD,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            bufsize=1,
            preexec_fn=os.setsid,
        )
    except Exception as exc:
        _log("ERROR: subprocess.Popen failed: " + repr(exc))
        return
    _proc["handle"] = proc
    reader = threading.Thread(target=_reader_loop, args=(proc,), daemon=True)
    reader.start()
    _proc["reader"] = reader
    _log("runner pid=%d (will load AudioLabOverlay; first HDMI frame in 30..60 s)."
         % proc.pid)


def _stop_runner_locked(reason):
    proc = _proc["handle"]
    if proc is None or proc.poll() is not None:
        _log("stop_runner (%s): no live runner." % reason)
        return
    pid = proc.pid
    _log("stop_runner (%s): SIGTERM pid=%d (runner mutes Pmod on exit)."
         % (reason, pid))
    try:
        os.killpg(os.getpgid(pid), signal.SIGTERM)
    except Exception as exc:
        _log("SIGTERM failed: " + repr(exc))
    deadline = time.time() + 6.0
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        time.sleep(0.1)
    if proc.poll() is None:
        _log("runner did not exit in 6 s; sending SIGKILL.")
        try:
            os.killpg(os.getpgid(pid), signal.SIGKILL)
        except Exception as exc:
            _log("SIGKILL failed: " + repr(exc))
    _log("runner stopped (rc=%s)." % proc.poll())


def start_runner():
    with _proc_lock:
        _start_runner_locked()


def stop_runner(reason="user"):
    with _proc_lock:
        _stop_runner_locked(reason)


def panic_mute():
    """SIGTERM the runner; its shutdown path writes Pmod MODE=3.

    If for some reason the runner is not alive (already crashed), shell
    out to `pmod_i2s2_mode.py --mode mute` so the DAC is still silenced.
    """
    with _proc_lock:
        if _is_running():
            _stop_runner_locked("panic")
            return
    _log("panic_mute: runner not alive -- shelling out to write MODE=3.")
    _run_mode_helper(["--mode", "mute"], label="panic_mute")


def _run_mode_helper(args, label="mode_helper"):
    if not os.path.exists(MODE_HELPER_PATH):
        _log("ERROR: " + MODE_HELPER_PATH + " missing on this PYNQ; "
             "deploy the latest scripts/ tree.")
        return None
    cmd = list(MODE_HELPER_PREFIX) + list(args)
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, timeout=20)
    except Exception as exc:
        _log(label + " failed: " + repr(exc))
        return None
    out = result.stdout.decode("utf-8", errors="replace")
    return result.returncode, out


def set_pmod_mode(mode_name):
    extra = []
    if mode_name == "loopback":
        extra.append("--confirm-loopback")
    res = _run_mode_helper(["--mode", mode_name] + extra,
                           label="set_pmod_mode")
    if res is None:
        return
    rc, out = res
    for line in out.splitlines():
        _log("mode: " + line)
    if rc != 0:
        _log("set_pmod_mode(%s) returned rc=%d" % (mode_name, rc))


def refresh_status():
    res = _run_mode_helper(["--read"], label="refresh_status")
    with status_out:
        clear_output(wait=True)
        if res is None:
            print("pmod_status helper failed (see log).")
            return
        rc, out = res
        if rc != 0:
            print("pmod_status helper rc=%d" % rc)
        print(out)
        print("")
        print("Wiring reminder:")
        print("  - Disconnect on-module Line Out <-> Line In jumper")
        print("  - External source -> Pmod Line In (LOW volume to start)")
        print("  - Pmod Line Out -> audio IF / speakers / headphone amp")


def show_command():
    _log("runner command: " + " ".join(shlex.quote(c) for c in RUNNER_CMD))
    _log("mode helper   : " + " ".join(shlex.quote(c) for c in
                                       MODE_HELPER_PREFIX) + " --mode {tone,loopback,dsp,mute} | --read | --clear")


# --- Widgets --------------------------------------------------------------
btn_start    = widgets.Button(description="Start HDMI GUI + Pmod DSP",
                              button_style="success",
                              layout=widgets.Layout(width="240px"))
btn_stop     = widgets.Button(description="Stop HDMI GUI",
                              layout=widgets.Layout(width="160px"))
btn_panic    = widgets.Button(description="Panic / Mute Pmod",
                              button_style="danger",
                              layout=widgets.Layout(width="180px"))
btn_set_dsp  = widgets.Button(description="Set Pmod mode 2 / DSP",
                              button_style="info",
                              layout=widgets.Layout(width="200px"))
btn_refresh  = widgets.Button(description="Refresh Pmod status",
                              layout=widgets.Layout(width="180px"))
btn_show_cmd = widgets.Button(description="Show command",
                              layout=widgets.Layout(width="140px"))

btn_start   .on_click(lambda _b: start_runner())
btn_stop    .on_click(lambda _b: stop_runner("user"))
btn_panic   .on_click(lambda _b: panic_mute())
btn_set_dsp .on_click(lambda _b: set_pmod_mode("dsp"))
btn_refresh .on_click(lambda _b: refresh_status())
btn_show_cmd.on_click(lambda _b: show_command())

header = widgets.HTML(
    "<h3>Pmod I2S2 HDMI GUI (mode 2 = ADC -&gt; AudioLab DSP -&gt; DAC, D49)</h3>"
    "<p style='color:#a00'><b>Disconnect</b> the on-module Line Out &lt;-&gt; Line In jumper. "
    "External source on Pmod Line In at LOW level. Pmod Line Out to a SEPARATE audio interface, "
    "speakers, or headphone amp. Encoder 0 selects effect / toggles ON, Encoder 1 selects knob "
    "(press+rotate cycles model), Encoder 2 changes value.</p>"
)

row_run    = widgets.HBox([btn_start, btn_stop, btn_panic])
row_mode   = widgets.HBox([btn_set_dsp, btn_refresh, btn_show_cmd])

ui = widgets.VBox([
    header,
    row_run,
    row_mode,
    widgets.HTML("<b>Runner log</b>"),
    log_out,
    widgets.HTML("<b>Pmod status snapshot</b> (Refresh to update)"),
    status_out,
])

display(ui)

# Cleanup orphans from previous cell executions before we touch
# anything else; if the user re-ran the cell, the previously
# spawned runner is no longer tracked by this kernel.
_kill_orphan_runners(label="cell-init")

# Auto-start so the cell is genuinely one-shot. Wrap in try/except so a
# missing path / sudo refusal does not nuke the widget UI; the user can
# still click Start manually after fixing the environment.
try:
    start_runner()
except Exception as _exc:
    _log("auto-start failed: " + repr(_exc))
